In [1]:
import os
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

# ========================
# PARTE 1: CONFIGURAÇÕES
# ========================
data_nome = datetime.now().strftime("%Y-%m-%d")
base_drive = f"/content/drive/MyDrive/WebScraping/FEITO/CARRETAO/COLETA_CARRETAO/{data_nome}"
pasta_imagens = os.path.join(base_drive, "imagens")
os.makedirs(pasta_imagens, exist_ok=True)

# Categorias e URLs
categorias_urls = {
    "Banheiros": "https://carretaoconstrucao.com.br/banheiros.html",
    "Cozinhas e Áreas de Serviço": "https://carretaoconstrucao.com.br/cozinhas-e-areas-de-servico.html",
    "Ferragens": "https://carretaoconstrucao.com.br/ferragens.html",
    "Iluminação": "https://carretaoconstrucao.com.br/iluminacao.html",
    "Materiais Hidráulicos": "https://carretaoconstrucao.com.br/materiais-hidraulicos.html",
    "Móveis": "https://carretaoconstrucao.com.br/moveis.html",
    "Pisos e Revestimentos": "https://carretaoconstrucao.com.br/pisos-e-revestimentos.html"
}

# ========================
# Função para coletar links de uma categoria
# ========================
def coletar_links_categoria(url):
    links = set()
    while url:
        print(f"📄 Coletando página: {url}")
        resp = requests.get(url)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        for a in soup.select("a.product-item-link"):
            href = a.get("href")
            if href and href.startswith("http"):
                links.add(href)

        next_link = soup.find("link", rel="next")
        if next_link:
            url = next_link.get("href")
            time.sleep(1)
        else:
            url = None
    return sorted(list(links))

# ========================
# PARTE 2: EXTRAÇÃO + SALVAMENTO PARCIAL
# ========================
def extrair_dados_produto(link, categoria, index):
    resp = requests.get(link)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    nome_tag = soup.find("h1", class_="page-title") or soup.find("h1", {"data-ui-id": "page-title-wrapper"})
    nome = nome_tag.get_text(strip=True) if nome_tag else None

    codigo = None
    sku_div = soup.find("div", class_="product attribute sku")
    if sku_div:
        value_div = sku_div.find("div", class_="value")
        if value_div:
            codigo = value_div.get_text(strip=True)
    if not codigo:
        codigo = f"{categoria}_produto_{index}"

    preco_tag = soup.find("span", {"data-price-type": "finalPrice"})
    preco_promocional = preco_tag.get("data-price-amount") if preco_tag else None

    descricao_tag = soup.find("div", class_="product attribute description")
    descricao = descricao_tag.get_text(strip=True) if descricao_tag else None

    imagem_tag = soup.find("meta", {"property": "og:image"})
    imagem = imagem_tag.get("content") if imagem_tag else None

    url_produto = soup.find("meta", {"property": "og:url"})
    url_produto = url_produto.get("content") if url_produto else link

    loja = "Carretão Construção"

    # Baixar imagem
    caminho_img = os.path.join(pasta_imagens, f"{codigo}.png")
    if not os.path.exists(caminho_img) and imagem not in [None, "N/A"]:
        try:
            img = requests.get(imagem, stream=True, timeout=10)
            if img.status_code == 200:
                with open(caminho_img, 'wb') as f:
                    for chunk in img.iter_content(1024):
                        f.write(chunk)
        except Exception as e:
            print(f"⚠️ Erro ao baixar imagem {codigo}: {e}")

    return {
        "Código": codigo,
        "Nome": nome,
        "Preço Promocional": preco_promocional,
        "Descrição": descricao,
        "Imagem": imagem,
        "URL": url_produto,
        "Loja": loja,
        "Categoria": categoria
    }

# ========================
# PARTE 3: LOOP PRINCIPAL POR CATEGORIA
# ========================
for categoria, url in categorias_urls.items():
    print(f"\n===== CATEGORIA: {categoria} =====")
    links = coletar_links_categoria(url)
    print(f"🔗 Total de produtos encontrados: {len(links)}")

    todos = []
    for i, link in enumerate(links, 1):
        try:
            dados = extrair_dados_produto(link, categoria, i)
            todos.append(dados)

            if i % 100 == 0 or i == len(links):
                df = pd.DataFrame(todos)
                colunas = ["Código", "Nome", "Preço Promocional", "Descrição", "Imagem", "URL", "Loja", "Categoria"]
                df = df[colunas]
                df.to_csv(os.path.join(base_drive, f"{categoria}_parcial_{i}.csv"), index=False, sep=";")
                df.to_excel(os.path.join(base_drive, f"{categoria}_parcial_{i}.xlsx"), index=False)
                print(f"💾 [{categoria}] Dados salvos com {i} produtos.")
                todos.clear()

        except Exception as e:
            print(f"❌ Erro no produto {i} da categoria {categoria}: {e}")

# ========================
# PARTE 4: UNIFICAÇÃO FINAL
# ========================
print("\n📁 Unificando todos os arquivos CSV...")
csvs = [f for f in os.listdir(base_drive) if f.endswith(".csv") and "FINAL" not in f]
df_final = pd.DataFrame()

for csv in csvs:
    df = pd.read_csv(os.path.join(base_drive, csv), sep=";")
    df_final = pd.concat([df_final, df], ignore_index=True)

csv_final = os.path.join(base_drive, f"carretao_FINAL_{data_nome}.csv")
excel_final = os.path.join(base_drive, f"carretao_FINAL_{data_nome}.xlsx")

df_final.to_csv(csv_final, index=False, sep=";")
df_final.to_excel(excel_final, index=False)

print(f"✅ CSV final salvo: {csv_final}")
print(f"✅ Excel final salvo: {excel_final}")



===== CATEGORIA: Banheiros =====
📄 Coletando página: https://carretaoconstrucao.com.br/banheiros.html
📄 Coletando página: https://carretaoconstrucao.com.br/banheiros.html?p=2
📄 Coletando página: https://carretaoconstrucao.com.br/banheiros.html?p=3
📄 Coletando página: https://carretaoconstrucao.com.br/banheiros.html?p=4
📄 Coletando página: https://carretaoconstrucao.com.br/banheiros.html?p=5
📄 Coletando página: https://carretaoconstrucao.com.br/banheiros.html?p=6
📄 Coletando página: https://carretaoconstrucao.com.br/banheiros.html?p=7
📄 Coletando página: https://carretaoconstrucao.com.br/banheiros.html?p=8
📄 Coletando página: https://carretaoconstrucao.com.br/banheiros.html?p=9
📄 Coletando página: https://carretaoconstrucao.com.br/banheiros.html?p=10
📄 Coletando página: https://carretaoconstrucao.com.br/banheiros.html?p=11
🔗 Total de produtos encontrados: 125
💾 [Banheiros] Dados salvos com 100 produtos.
💾 [Banheiros] Dados salvos com 125 produtos.

===== CATEGORIA: Cozinhas e Áreas de 